# ZONOS3-Mini — multilingual TTS and voice clone on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/ZONOS3-Mini-voice-clone-tts.ipynb)

[`Zyphra/ZONOS3-Mini-hf`](https://huggingface.co/Zyphra/ZONOS3-Mini-hf) is Zyphra's compact **text-to-speech** model packaged for Transformers (Apache-2.0, not gated). The Hub file holds **~1.04B** parameters (`model.safetensors` **2.4 GB**). The [card](https://huggingface.co/Zyphra/ZONOS3-Mini-hf) describes **~100 million active parameters per token** and an **800 million** backbone (latent MoE, 64 experts, top-2). Output is **44.1 kHz** mono. **43 languages**, **27 release voices** in `voices.json` / `voices.safetensors`, and cloning through the bundled MECT-B2 speaker encoder. The native weights live at [`Zyphra/ZONOS3-Mini`](https://huggingface.co/Zyphra/ZONOS3-Mini).

**Why this model.** One Transformers checkpoint covers a 43-language catalog, named release voices, and clip cloning, without a separate codec download. Inference is `AutoModel.from_pretrained(..., trust_remote_code=True)` then `model.generate(...)` — the [Usage](https://huggingface.co/Zyphra/ZONOS3-Mini-hf#usage) section, not a `text-to-speech` pipeline (the card says that pipeline tokenizes first and skips text normalization).

This is the free-Colab Transformers path. [Silencio-2](Silencio-2-tts-voice-clone.ipynb) is tiny English with tags. [Phonon 31M](phonon-31m-en-streaming-tts.ipynb) is tinier English CPU TTS. [Audio8](Audio8-TTS-Realistic-Air-voice-clone.ipynb) is a different 0.6B DualAR stack. [Maya 2 Native](Maya2-Native-indic-tts.ipynb) is 2.50B Indic + English. [Swar](Swar-voice-clone-tts.ipynb) is Nepali/English F5-TTS. [Ming-3.0-tts 0.5B](Ming-3.0-tts-0.5B.ipynb) is another 0.5B stack. [Drama 3](Drama-3-tts-voice-clone.ipynb) needs a Colab Pro A100.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/ZONOS3-Mini-voice-clone-tts.md) for when this notebook is a good fit and when it is not.

**Licence:** [Apache-2.0](https://huggingface.co/Zyphra/ZONOS3-Mini-hf) on the weights. Remote code is ported from the [ZONOS3-Mini repository](https://github.com/Zyphra/ZONOS3-Mini) (MIT) with adapted AntSpeaker, Transformers Qwen3-Next, and torchaudio pieces — see [`NOTICE`](https://huggingface.co/Zyphra/ZONOS3-Mini-hf/blob/main/NOTICE).

**Consent:** clone a voice only with its owner's permission. Do not impersonate people or deceive listeners. The clone demo below reuses a clip **this notebook just synthesised**. Label synthetic speech as synthetic where it could be mistaken for a person.

**Runtime:** *Runtime → Change runtime type → T4 GPU*. A free T4 has ~15–16 GB and **no native bfloat16**. The checkpoint's LM is stored as bfloat16; the card says Transformers 5 loads it as stored and 4.x loads float32 unless you pass a dtype. This notebook loads the LM as **float16** on compute capability &lt; 8.0 and **bfloat16** on L4 / A100. The card keeps the codec decoder and speaker encoder in **float32**. First run downloads ~2.4 GB `model.safetensors` plus the tokenizer and 27 voices. Short clips should stay well under the T4's 15 GB; the last cell prints measured peak VRAM.

**Not in this notebook (card):** speech editing, steering controls and accents, LACI, phoneme input, tag CFG, streaming decode, and the codec encoder. Those live in the [native runtimes](https://github.com/Zyphra/ZONOS3-Mini). Do not call the Transformers `text-to-speech` pipeline.

**Links:** [Transformers card](https://huggingface.co/Zyphra/ZONOS3-Mini-hf) · [base card](https://huggingface.co/Zyphra/ZONOS3-Mini) · [blog](https://www.zyphra.com/our-work/zonos3-mini) · [tech report](https://arxiv.org/abs/2611.16525) · [GitHub](https://github.com/Zyphra/ZONOS3-Mini) · [announcement](https://x.com/ZyphraAI/status/2119914895861076151) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/ZONOS3-Mini-voice-clone-tts.md)


## 0. Early GPU check

Print `torch.cuda.is_available()`, the GPU name, and free VRAM before installing anything. Need a CUDA GPU (T4 is the free-Colab target). A restart after the install cell drops this state; section 2 prints it again and sets `DEVICE` / `DTYPE`.


In [ ]:
import shutil

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 on this GPU; this notebook will load the LM as float16.")
else:
    print("GPU name: none")
    print("VRAM free: n/a")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 1. Install the card's Transformers stack (keep Colab's torch)

The [model card](https://huggingface.co/Zyphra/ZONOS3-Mini-hf#usage) install is:

```bash
pip install "transformers>=4.57" torch soundfile
pip install zyphra-textnorm   # optional, recommended: reads numbers, dates, currencies and units aloud
```

Current Colab already has a CUDA `torch`. This cell installs the remaining pins and does **not** replace Colab's torch.

Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`. Do **not** install `flash-attn`. `zyphra-textnorm` is the card's optional normalizer (Rust, no `pynini`); without it the model still runs and only collapses line breaks, with a one-time warning.


In [ ]:
%pip install -q "transformers>=4.57" soundfile zyphra-textnorm

# Card: pip install "transformers>=4.57" torch soundfile
# Keep Colab's CUDA torch. Use %pip, not python -I.

import sys

import torch
import transformers

print("python", sys.version.split()[0])
print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__)
try:
    import zyphra_textnorm

    print("zyphra_textnorm", getattr(zyphra_textnorm, "__version__", "ok"))
except ImportError:
    print("zyphra_textnorm: not importable (generate still works; numbers/dates will not be spoken out)")


> **Restart only if transformers was upgraded.** If the install cell printed a version **below 4.57**, use *Runtime → Restart session*, then continue from the next cell (do not rerun the install unless it failed). Colab often already has Transformers 5.x, which satisfies `>=4.57` — skip the restart in that case. The card notes Transformers 5 loads the LM as stored (bfloat16); this notebook still passes an explicit dtype so a T4 does not run emulated bfloat16.


## 2. Runtime check and T4 dtype

Need a CUDA GPU. Print `torch.cuda.is_available()`, the device name, and free VRAM again (a restart would have dropped the first cell's state). Pick **float16** on a T4 and **bfloat16** only when the GPU has native BF16 (sm80+). The codec and speaker encoder stay float32 (card / `_keep_in_fp32_modules_strict`).


In [ ]:
import os
import shutil

import torch
import transformers
from packaging.version import Version

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

tf_ver = Version(transformers.__version__)
if tf_ver < Version("4.57"):
    raise SystemExit(
        f"transformers {transformers.__version__} is loaded; ZONOS3-Mini-hf needs >=4.57. "
        "Run the install cell, then Runtime → Restart session if that cell upgraded transformers."
    )

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime → Change runtime type → T4 GPU, then rerun from the GPU-check cell.")

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

free_b, total_b = torch.cuda.mem_get_info()
major, minor = torch.cuda.get_device_capability()
print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")

# Card CUDA weights are bfloat16. T4 has no native BF16 — use float16 there.
NATIVE_BF16 = major >= 8
DEVICE = "cuda"
DTYPE = torch.bfloat16 if NATIVE_BF16 else torch.float16
MODEL_ID = "Zyphra/ZONOS3-Mini-hf"
print("DEVICE", DEVICE, "| DTYPE", DTYPE)
if not NATIVE_BF16:
    print("No native bfloat16; loading the LM as float16 (codec and speaker encoder stay float32).")


## 3. Load ZONOS3-Mini-hf (`trust_remote_code`)

Same API as the [model card](https://huggingface.co/Zyphra/ZONOS3-Mini-hf#usage):

```python
from transformers import AutoModel

model = AutoModel.from_pretrained("Zyphra/ZONOS3-Mini-hf", trust_remote_code=True)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
```

`AutoModelForTextToWaveform` loads the same class. Review the custom files in the Hub repo (`modeling_zonos3.py`, `configuration_zonos3.py`) before you `trust_remote_code`. The first run downloads `model.safetensors` (~2.4 GB), the SuperBPE tokenizer, `voices.json`, and `voices.safetensors`.

This cell adds `dtype=DTYPE` so a T4 does not load the LM as bfloat16. If a later waveform is non-finite, rerun this cell with `DTYPE = torch.float32` (still fits ~15 GB).


In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    dtype=DTYPE,
)
model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
voices = model.voices()
print("repo:", MODEL_ID)
print("class:", type(model).__name__)
print(f"parameters: {n_params:,} ({n_params / 1e9:.3f}B; Hub ~1.04B)")
print("sampling_rate:", model.config.sampling_rate)
print(f"release voices ({len(voices)}):", ", ".join(sorted(voices)))
print("LM dtype:", next(model.lm.parameters()).dtype)
print("codec dtype:", next(model.codec.parameters()).dtype)
print("speaker_encoder dtype:", next(model.speaker_encoder.parameters()).dtype)
print("device:", next(model.parameters()).device)
if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    print(
        f"VRAM after load: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"free / total {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB"
    )


## 4. Built-in release voice — English

Card snippet: `model.generate(..., voice="Savannah", seed=0)`. Savannah is the American English female release voice. `generate` returns `waveform` (float32 samples), `sampling_rate` (44100), `seed`, and `codes`. Play inline and write a WAV.

`max_new_frames=150` is a 10 s cap (the card: 15 frames per second). Short lines stop on EOS earlier.


In [ ]:
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

# Card example — a release voice speaking English.
out = model.generate(
    "Hello! This is ZONOS3-Mini running in transformers.",
    voice="Savannah",
    seed=0,
    max_new_frames=150,
)

audio = out.waveform.float().cpu().numpy()
if audio.size == 0 or not np.isfinite(audio).all():
    raise SystemExit("English decode returned empty or non-finite audio.")

sr = int(out.sampling_rate)
sf.write("zonos3-en.wav", audio, sr)
print(
    f"English / Savannah → zonos3-en.wav | {len(audio) / sr:.2f} s | {sr} Hz | "
    f"seed {out.seed} | codes {tuple(out.codes.shape)}"
)
display(Audio(audio, rate=sr))


## 5. Same voice, another language

`language` selects **text normalization**, not a language token in the prompt (card). Spanish (`es`) is one of the languages `zyphra_textnorm` covers. Same release voice, Spanish text.


In [ ]:
out = model.generate(
    "Hola, este es ZONOS3-Mini hablando en español.",
    voice="Savannah",
    language="es",
    seed=0,
    max_new_frames=150,
)

audio = out.waveform.float().cpu().numpy()
if audio.size == 0 or not np.isfinite(audio).all():
    raise SystemExit("Spanish decode returned empty or non-finite audio.")

sr = int(out.sampling_rate)
sf.write("zonos3-es.wav", audio, sr)
print(
    f"Spanish / Savannah → zonos3-es.wav | {len(audio) / sr:.2f} s | {sr} Hz | "
    f"seed {out.seed} | codes {tuple(out.codes.shape)}"
)
display(Audio(audio, rate=sr))


## 6. Voice clone from a clip this notebook just made

Card clone path: `speaker_audio` + `speaker_sample_rate` through the speaker encoder (this Transformers checkpoint has no codec encoder, so raw clips clone as embeddings). There is no public third-party clip here — the reference is `zonos3-en.wav` from demo 1. That keeps the audio license-clean and avoids cloning a real person.

Only clone a voice you have permission to use.


In [ ]:
# Card clone snippet: sf.read(..., always_2d=True) then speaker_audio=wav.T
wav, sr = sf.read("zonos3-en.wav", dtype="float32", always_2d=True)
print("reference (model-made, no real speaker): zonos3-en.wav", wav.shape, sr, "Hz")
display(Audio("zonos3-en.wav"))

out = model.generate(
    "Cloned from a clip.",
    speaker_audio=wav.T,
    speaker_sample_rate=sr,
    seed=1,
    max_new_frames=150,
)

clone = out.waveform.float().cpu().numpy()
if clone.size == 0 or not np.isfinite(clone).all():
    raise SystemExit("Clone decode returned empty or non-finite audio.")

sf.write("zonos3-clone.wav", clone, int(out.sampling_rate))
print(
    f"clone from zonos3-en.wav → zonos3-clone.wav | {len(clone) / out.sampling_rate:.2f} s | "
    f"{out.sampling_rate} Hz | seed {out.seed}"
)
display(Audio(clone, rate=int(out.sampling_rate)))

if torch.cuda.is_available():
    print(
        f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )


## 7. Caveats and cleanup

- **Licence.** Apache-2.0 on the weights. Keep the [NOTICE](https://huggingface.co/Zyphra/ZONOS3-Mini-hf/blob/main/NOTICE) attribution (ZONOS3-Mini MIT remote code, AntSpeaker, Transformers Qwen3-Next, torchaudio).
- **Install.** Card: `pip install "transformers>=4.57" torch soundfile` and optional `zyphra-textnorm`. Keep Colab's torch. Use `%pip`, not `python -I`. Restart only if transformers was below 4.57 before the install.
- **API.** `AutoModel.from_pretrained(..., trust_remote_code=True)` then `model.generate(text, voice=..., language=..., speaker_audio=..., seed=...)`. Do **not** use the Transformers `text-to-speech` pipeline. `AutoModelForTextToWaveform` is the same class.
- **T4 / dtype.** Checkpoint LM is bfloat16. This notebook uses **float16** on sm75 (no native BF16). Codec and speaker encoder stay float32. If a waveform is non-finite, rerun the load cell with `DTYPE = torch.float32`.
- **Languages.** 43 request codes on the card. `language` picks the normalizer, not a language id in the prompt. Hindi and most Indic languages pass through as written.
- **Cloning.** Clean reference, up to 60 s. This notebook clones a clip it just synthesised. Do not upload a recording of a real person without consent. Reuse an embedding with `model.embed_speaker(wav, sr)` if you generate several takes of the same speaker.
- **Not in this Transformers build.** Speech editing, steering / accents, LACI, phoneme input, tag CFG, streaming decode, codec encoder — native runtimes only.
- **Not Silencio-2, not Maya 2, not Swar, not Audio8, not Ming-3.0, not Drama 3.** Those are different stacks and licences.

Free the GPU before you open another notebook. Peak VRAM below is `torch.cuda.max_memory_allocated` for this session.


In [ ]:
import gc

if torch.cuda.is_available():
    print(f"before cleanup: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

del model, out, audio, clone, wav
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    print(
        f"after cleanup: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"free / total {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB"
    )
else:
    print("CPU runtime — nothing to empty on the GPU.")
